In [5]:
# 필요한 라이브러리 설치:
!pip install google-play-scraper
!pip install konlpy
!pip uninstall JPype1-py3
!pip install JPype1


In [18]:
import re
import json
from google_play_scraper import reviews, Sort
from konlpy.tag import Okt
from collections import Counter

def fetch_reviews(app_id, total_reviews=1000):
    """Google Play Store에서 total_reviews 수 만큼 리뷰를 가져옵니다."""
    all_reviews = []
    count = 100  # 한 번에 요청할 리뷰 개수
    token = None

    while len(all_reviews) < total_reviews:
        result, token = reviews(
            app_id,
            lang='ko',
            country='kr',
            sort=Sort.NEWEST,
            count=count,
            continuation_token=token
        )
        all_reviews.extend(result)
        if token is None:
            break  # 더 이상 리뷰가 없으면 종료
    return all_reviews[:total_reviews]

def analyze_reviews(review_list):
    """리뷰 목록에서 텍스트를 결합하여 명사를 추출한 후 빈도 상위 10개 단어를 반환합니다."""
    okt = Okt()
    # 모든 리뷰의 텍스트를 하나의 문자열로 결합
    all_text = " ".join([review['content'] for review in review_list])
    # 한글과 공백을 제외한 문자 제거
    all_text = re.sub(r'[^가-힣\s]', '', all_text)
    
    # 명사 추출
    nouns = okt.nouns(all_text)
    
    # 불용어 처리 (필요에 따라 추가 가능)
    stopwords = set(['이', '그', '저', '것', '들', '의', '있', '하', '되', '수', '때문', '절대', '원래', '관련', '아예', '읍니', '만하','에드','리기'])
    # 한 글자 단어 및 불용어 제거
    nouns = [noun for noun in nouns if noun not in stopwords and len(noun) > 1]
    
    counter = Counter(nouns)
    top_10 = counter.most_common(10)
    return top_10

def group_reviews_by_rating(review_list, target_count=10):
    """
    리뷰 목록을 평점(1~5점)별로 그룹화하여 각 평점에서 최대 target_count개의 리뷰만 반환합니다.
    """
    grouped = {1: [], 2: [], 3: [], 4: [], 5: []}
    for review in review_list:
        score = review.get('score')
        if score in grouped and len(grouped[score]) < target_count:
            grouped[score].append(review)
    return grouped

if __name__ == "__main__":
    # 중고나라 앱의 패키지명 (실제 앱의 패키지명으로 변경하세요)
    app_id = 'com.elz.secondhandstore'
    
    print("리뷰를 수집 중입니다...")
    reviews_data = fetch_reviews(app_id, total_reviews=2000)
    total_reviews = len(reviews_data)
    print(f"{total_reviews}개의 리뷰를 수집했습니다.")
    
    # 전체 리뷰에서 평점별 리뷰 개수를 계산
    overall_counts = Counter(review['score'] for review in reviews_data)
    
    print("\n전체 리뷰 평점별 개수:")
    for score in range(1, 6):
        count_for_score = overall_counts.get(score, 0)
        print(f"{score}점 리뷰: {count_for_score}개")
    
    print(f"\n1점대 리뷰 총 개수: {overall_counts.get(1, 0)}개")
    
    # 평점별로 최대 10개씩 리뷰 그룹화 (분석용)
    grouped_reviews = group_reviews_by_rating(reviews_data, target_count=2000)
    
    analysis_result = {}
    
    # 각 평점 그룹별로 리뷰 분석 수행 및 결과 저장
    for score in sorted(grouped_reviews.keys()):
        reviews_for_score = grouped_reviews[score]
        print(f"\n평점 {score}점 리뷰 {len(reviews_for_score)}개 분석:")
        if reviews_for_score:
            top_words = analyze_reviews(reviews_for_score)
            print("가장 많이 나온 단어 TOP 10:")
            for rank, (word, freq) in enumerate(top_words, start=1):
                print(f"{rank}위: {word} - {freq}회")
            # JSON 출력을 위해 tuple을 dict 형태로 변환
            analysis_result[str(score)] = {
                "review_count": len(reviews_for_score),
                "top_words": [{"word": word, "frequency": freq} for word, freq in top_words]
            }
        else:
            print("해당 평점의 리뷰가 없습니다.")
            analysis_result[str(score)] = {
                "review_count": 0,
                "top_words": []
            }
    
    # 최종 결과를 JSON 형태로 구성
    result_data = {
        "total_reviews": total_reviews,
        "overall_counts": {str(score): overall_counts.get(score, 0) for score in range(1, 6)},
        "analysis": analysis_result
    }
    


리뷰를 수집 중입니다...
2000개의 리뷰를 수집했습니다.

전체 리뷰 평점별 개수:
1점 리뷰: 485개
2점 리뷰: 58개
3점 리뷰: 126개
4점 리뷰: 183개
5점 리뷰: 1148개

1점대 리뷰 총 개수: 485개

평점 1점 리뷰 485개 분석:
가장 많이 나온 단어 TOP 10:
1위: 중고나라 - 113회
2위: 카페 - 104회
3위: 거래 - 98회
4위: 판매 - 91회
5위: 어플 - 90회
6위: 네이버 - 67회
7위: 채팅 - 62회
8위: 인증 - 55회
9위: 연동 - 53회
10위: 계속 - 50회

평점 2점 리뷰 58개 분석:
가장 많이 나온 단어 TOP 10:
1위: 카페 - 14회
2위: 판매 - 13회
3위: 중고나라 - 12회
4위: 상품 - 9회
5위: 물건 - 9회
6위: 사진 - 9회
7위: 완료 - 9회
8위: 사용 - 8회
9위: 어플 - 6회
10위: 거래 - 6회

평점 3점 리뷰 126개 분석:
가장 많이 나온 단어 TOP 10:
1위: 판매 - 39회
2위: 카페 - 38회
3위: 중고나라 - 28회
4위: 사진 - 22회
5위: 거래 - 22회
6위: 완료 - 18회
7위: 상품 - 17회
8위: 연동 - 17회
9위: 수정 - 16회
10위: 채팅 - 16회

평점 4점 리뷰 183개 분석:
가장 많이 나온 단어 TOP 10:
1위: 거래 - 23회
2위: 사용 - 17회
3위: 상품 - 14회
4위: 중고나라 - 13회
5위: 중고 - 12회
6위: 안전 - 9회
7위: 처음 - 9회
8위: 판매 - 7회
9위: 알림 - 6회
10위: 보기 - 6회

평점 5점 리뷰 1148개 분석:
가장 많이 나온 단어 TOP 10:
1위: 거래 - 105회
2위: 사용 - 76회
3위: 중고나라 - 64회
4위: 최고 - 44회
5위: 판매 - 43회
6위: 안전 - 42회
7위: 아주 - 39회
8위: 이용 - 37회
9위: 상품 - 31회
10위: 중고 - 31회


In [19]:
    print("\nJSON 형태 결과:")
    print(json.dumps(result_data, ensure_ascii=False, indent=2))


JSON 형태 결과:
{
  "total_reviews": 2000,
  "overall_counts": {
    "1": 485,
    "2": 58,
    "3": 126,
    "4": 183,
    "5": 1148
  },
  "analysis": {
    "1": {
      "review_count": 485,
      "top_words": [
        {
          "word": "중고나라",
          "frequency": 113
        },
        {
          "word": "카페",
          "frequency": 104
        },
        {
          "word": "거래",
          "frequency": 98
        },
        {
          "word": "판매",
          "frequency": 91
        },
        {
          "word": "어플",
          "frequency": 90
        },
        {
          "word": "네이버",
          "frequency": 67
        },
        {
          "word": "채팅",
          "frequency": 62
        },
        {
          "word": "인증",
          "frequency": 55
        },
        {
          "word": "연동",
          "frequency": 53
        },
        {
          "word": "계속",
          "frequency": 50
        }
      ]
    },
    "2": {
      "review_count": 58,
      "top_words": [
  

In [12]:
# 전체 리뷰 출력
for idx, r in enumerate(reviews_data, start=1):
    score = r.get('score')
    content = r.get('content').replace('\n',' ')
    print(f"{idx:4d}. [{score}점] {content}")

   1. [4점] 좋은 중고거래 추천합니다
   2. [1점] 갤럭시 노트9 사용자 입니다 업데이트 이후 앱 광고부분에 노이즈 이미지가 출력되며 랙이 많이 생기며 사용에 큰 불편함이 있습니다 화면 스크롤 몇번이면 앱사용이 중단되며 랙때문에 이미지조차 확인이 잘안됩니다
   3. [1점] 계속 화면 멈추고, 아무버튼도 눌러지지 않습니다. 되는가 싶다가도 다시 같은 현상 반복됩니다. 앱 제거하고 다시 깔아도 마찬가지 입니다.
   4. [4점] 키워드 알림이 떠서 들어가면 창은 뜨는데 그뒤로 버튼조작이 안먹혀서 앱을 껐다가 다시 그 판매글까지 가야하는 불편함이 있습니다. 원래 안그랬는데 어느순간부터 그러네요.
   5. [4점] 잘 사용하고 있습니다. 다만 가게소개란이 너무 작습니다. 글자 수 제한 늘려주세요! 그리고 상품 숨기기 기능이 있었으면 좋겠습니다.
   6. [5점] 매우 편리
   7. [3점] 사진등록이 왜 안되나요
   8. [5점] 번장보다 나음
   9. [1점] 한 상품당 끌올을 10회로 제한 하는것은 진짜 바보같은 짓입니다. 특히 의류는 올린지 1년뒤에도 너무 만족하면서 구매하는 구매자가 나타나는 경우도 종종있어요. 10회 끌올해서 안팔린다고 상품이 매력이 없다는 이야기가 절대 아니에요. 물건 올리는게 은근히 번거로운데 끌올 10회로 제한있으니 걍 번개장터 올리고 맙니다. 특히 좋은물건 많은 사람믈은요. 10일동안 끌올하면 더이상 끌올 못한다면 처음부터 시작을 안하는게 나음. 번개장터는 하루 10회제한있지만 상품당 끌올은 제한이 없습니다. 이것이 영리한 끌올 정책이라구요.
  10. [5점] 굿 입니다
  11. [5점] 판매 편함
  12. [5점] 처음이라 아직은 어리벙벙한데 곧 적응할것같네요 ~^^*
  13. [5점] 사기 당했는데 어떻게 하죠?
  14. [1점] 최악의 어플.. 채팅 한 줄 쓰는 동안에도 몇번의 네트워크 문제 알림이 뜨고 전송도 잘 안되고..
  15. [1점] 아니 왜 채팅알림 다켜놨는데 맨날 안오는거임? ㅡㅡ